### Imports

In [32]:
import sys
sys.path.append("..")

import pandas as pd
from collections import Counter
from scrapers import barbora, rimi
from scrapers.common import save_rows

importlib.reload(rimi)
importlib.reload(barbora)

<module 'scrapers.barbora' from '/Users/rokasslivinskas/Projects/end-to-end-AI/eat-easy/notebooks/../scrapers/barbora.py'>

### Sanity check

In [33]:
test_rows = barbora.scrape(
    "c/LVOK/mesa-zuvis-ir-kulinarija/sviezia-mesa-ir-paukstiena",
    max_pages=1
)

test_df = pd.DataFrame(test_rows)

print("Products:", len(test_df))
print("Columns:", len(test_df.columns))
print("Units:", test_df["unit"].unique())

display(test_df.head())

barbora page 1: 60 products, 60 new (category total: 306)
Products: 60
Columns: 15
Units: <StringArray>
['kg']
Length: 1, dtype: str


,store,id,name,brand,price,regular_price,loyalty_card_required,on_promo,unit_price,regular_unit_price,unit,category,category_id,image,url
0,barbora,000000000001320718,"Lietuviška smulk. kiauliena, rieb. ne did. kai...",WELL DONE,1.59,1.59,False,False,3.98,3.98,kg,mesa-zuvis-ir-kulinarija/sviezia-mesa-ir-pauks...,TJC6,https://cdn.barbora.lt/products/8f11d950-eb04-...,https://barbora.lt/produktai/sviezia-lietuvisk...
1,barbora,000000000001278140,Lietuviška šviežia viščiukų broilerių filė WEL...,WELL DONE,3.99,3.99,False,False,7.98,7.98,kg,mesa-zuvis-ir-kulinarija/sviezia-mesa-ir-pauks...,AFE0,https://cdn.barbora.lt/products/1b71f9be-3c06-...,https://barbora.lt/produktai/lietuviska-sviezi...
2,barbora,000000000001377915,Šviežia pjaustyta kiaulienos nugarinė WELL DON...,WELL DONE,2.49,2.49,False,False,6.92,6.92,kg,mesa-zuvis-ir-kulinarija/sviezia-mesa-ir-pauks...,R1J2,https://cdn.barbora.lt/products/de4b5bad-699d-...,https://barbora.lt/produktai/sviezia-pjaustyta...
3,barbora,000000000001390231,"Smulkinta jautiena WELL DONE, rieb. ne did. ne...",WELL DONE,4.29,4.29,False,False,10.73,10.73,kg,mesa-zuvis-ir-kulinarija/sviezia-mesa-ir-pauks...,TJC6,https://cdn.barbora.lt/products/b719beb4-d69f-...,https://barbora.lt/produktai/sviezia-smulkinta...
4,barbora,000000000001309590,"Šv. vištien. faršas VILNIAUS PAUKŠTYNAS, rieb....",VILNIAUS PAUKŠTYNAS,2.09,2.09,False,False,5.23,5.23,kg,mesa-zuvis-ir-kulinarija/sviezia-mesa-ir-pauks...,TJC6,https://cdn.barbora.lt/products/71c6c8cf-4262-...,https://barbora.lt/produktai/sv-vistien-farsas...


### Lets try testing other categories

In [3]:
test_rimi = rimi.scrape(
    "SH-11",
    "pieno-produktai-ir-kiausiniai",
    max_pages=1,
)

print("Products:", len(test_rimi))
print("Units:", Counter(row.get("unit") for row in test_rimi))
print("Missing unit prices:",
      sum(row.get("unit_price") is None for row in test_rimi))


rimi page 1: 40 cards, 40 new
Products: 40
Units: Counter({'kg': 27, 'l': 8, 'vnt': 5})
Missing unit prices: 0


In [4]:
import pandas as pd

df = pd.DataFrame(test_rimi)

df[df["unit"].isin(["l", "vnt"])][
    ["name", "price", "unit_price", "unit"]
]

,name,price,unit_price,unit
0,"Kavos gėrimas HOCHWALD EISKAFFEE, 1,5%, 500ml",1.29,2.58,l
1,"Šokoladinis pieno gėrimas HOCHWALD,1,5%,500ml",1.39,2.78,l
2,"Šalta kava LANDESSA CAPPUCCINO, 230 ml",1.79,7.78,l
3,"Šalta kava LANDESSA LATTE MACCIATO, 230 ml",1.79,7.78,l
6,"Laisvai laikomų vištų kiaušiniai RIMI,10 vnt.",2.95,0.30,vnt
10,"Ekol. vištų kiaušin.Rimi Eco,M/L,10 vnt.",3.49,0.35,vnt
12,"Kaimiški kiaušiniai RIMI (M dydis), 10 vnt.",1.89,0.19,vnt
19,"Vištų kiauš. RIMI SMART, A/M, Nr.2, 10 vnt.",2.19,0.22,vnt
22,"Majonezas HELLMANN'S ORIGINAL, 855 ml",7.49,8.76,l
25,"Natūralus DVARO pienas, 3,5 % rieb., 2 l",2.79,1.40,l


### Scrape all

Barbora took around 6mins

Rimi took 9 mins

In [6]:
# Define categories

barbora_categories = {
    "fresh-meat": "c/LVOK/mesa-zuvis-ir-kulinarija/sviezia-mesa-ir-paukstiena",
    "produce": "c/KP8A/darzoves-ir-vaisiai",
    "dairy-eggs": "c/JZV8/pieno-gaminiai-kiausiniai-ir-majonezas",
    "bread": "c/Z1ZC/duonos-gaminiai-ir-konditerija",
    "frozen": "c/4X7M/saldytas-maistas",
    "fish": "c/3B5R/mesa-zuvis-ir-kulinarija/sviezia-zuvis-juru-gerybes-ir-zuvu-gaminiai",
    "canned": "c/L6WK/bakaleja/konservuotas-maistas",
    "oils": "c/DM1F/bakaleja/aliejus-ir-actas",
    "sauces": "c/FC2O/bakaleja/padazai-ir-konservuotos-uztepeles",
    "international": "c/51FL/bakaleja/pasaulio-virtuves",
    "pasta": "c/4UCM/bakaleja/makaronai",
    "grains": "c/HQUT/bakaleja/kruopos",
    "flour": "c/W9PQ/bakaleja/miltai-ir-ju-misiniai",
    "baking": "c/XHK3/bakaleja/cukrus-druska-ir-kepimo-priedai",
    "spices": "c/FZRO/bakaleja/prieskoniai-marinatai-ir-sultiniai",
    "nuts-seeds": "c/GB05/bakaleja/riesutai-seklos-dziovinti-vaisiai-ir-uogos",
}

rimi_categories = {
    "meat-and-fish": ("SH-9", "mesa-ir-zuvis-"),
    "produce": ("SH-15", "vaisiai-darzoves-ir-geles"),
    "dairy-eggs": ("SH-11", "pieno-produktai-ir-kiausiniai"),
    "pantry": ("SH-2", "bakaleja"),
    "bread": ("SH-3", "duonos-gaminiai-"),
    "frozen": ("SH-13", "saldytas-maistas"),
    "plant-based": ("SH-77", "augaliniai-produktai"),
}


In [34]:

from scrapers import barbora, rimi
from scrapers.common import save_rows

barbora_summary = {}

for name, category_path in barbora_categories.items():

    print(f"\nScraping Barbora: {name}")

    rows = barbora.scrape(
        category_path,
        max_pages=100
    )

    barbora_summary[name] = len(rows)

    if rows:
        save_rows(rows, "barbora", name)
        print(f"Saved {len(rows)} products")



Scraping Barbora: fresh-meat
barbora page 1: 60 products, 60 new (category total: 306)
barbora page 2: 60 products, 60 new (category total: 306)
barbora page 3: 60 products, 60 new (category total: 306)
barbora page 4: 60 products, 60 new (category total: 306)
barbora page 5: 60 products, 60 new (category total: 306)
barbora page 6: 6 products, 6 new (category total: 306)
Saved 306 products

Scraping Barbora: produce
barbora page 1: 60 products, 60 new (category total: 313)
barbora page 2: 60 products, 60 new (category total: 313)
barbora page 3: 60 products, 60 new (category total: 313)
barbora page 4: 60 products, 60 new (category total: 313)
barbora page 5: 60 products, 60 new (category total: 313)
barbora page 6: 13 products, 13 new (category total: 313)
Saved 313 products

Scraping Barbora: dairy-eggs
barbora page 1: 60 products, 60 new (category total: 1187)
barbora page 2: 60 products, 60 new (category total: 1187)
barbora page 3: 60 products, 60 new (category total: 1187)
barb

In [35]:
display(
    pd.Series(barbora_summary, name="Products")
    .to_frame()
)

print("Total products:", sum(barbora_summary.values()))

,Products
fresh-meat,306
produce,313
dairy-eggs,1187
bread,585
frozen,603
fish,355
canned,417
oils,141
sauces,319
international,184


Total products: 5861


In [30]:
rimi_summary = {}

for name, (code, slug) in rimi_categories.items():
    print(f"\nScraping Rimi: {name}")

    rows = rimi.scrape(code, slug, max_pages=100)

    rimi_summary[name] = len(rows)

    if rows:
        save_rows(rows, "rimi", name)
        print(f"Saved {len(rows)} products")


Scraping Rimi: meat-and-fish
rimi page 1: 40 cards, 40 new
rimi page 2: 40 cards, 40 new
rimi page 3: 40 cards, 40 new
rimi page 4: 40 cards, 40 new
rimi page 5: 40 cards, 40 new
rimi page 6: 40 cards, 40 new
rimi page 7: 40 cards, 40 new
rimi page 8: 40 cards, 40 new
rimi page 9: 40 cards, 40 new
rimi page 10: 40 cards, 40 new
rimi page 11: 40 cards, 40 new
rimi page 12: 40 cards, 40 new
rimi page 13: 40 cards, 40 new
rimi page 14: 40 cards, 40 new
rimi page 15: 40 cards, 40 new
rimi page 16: 40 cards, 40 new
rimi page 17: 40 cards, 40 new
rimi page 18: 40 cards, 40 new
rimi page 19: 40 cards, 40 new
rimi page 20: 40 cards, 40 new
rimi page 21: 40 cards, 40 new
rimi page 22: 40 cards, 40 new
rimi page 23: 40 cards, 40 new
rimi page 24: 34 cards, 34 new
rimi page 25: 0 cards, 0 new
Saved 954 products

Scraping Rimi: produce
rimi page 1: 40 cards, 40 new
rimi page 2: 40 cards, 40 new
rimi page 3: 40 cards, 40 new
rimi page 4: 40 cards, 40 new
rimi page 5: 40 cards, 40 new
rimi page 6: 

### Doublecheck rerun of rimi

In [31]:
import pandas as pd

display(
    pd.Series(rimi_summary, name="Products")
    .to_frame()
)

print("Total products:", sum(rimi_summary.values()))

,Products
meat-and-fish,954
produce,388
dairy-eggs,923
pantry,2653
bread,292
frozen,501
plant-based,108


Total products: 5819


### Summarize everything 

In [9]:
import pandas as pd

summary = pd.DataFrame({
    "Barbora": pd.Series(barbora_summary),
    "Rimi": pd.Series(rimi_summary),
})

summary = summary.fillna(0).astype(int)
summary.loc["TOTAL"] = summary.sum()

display(summary)


,Barbora,Rimi
baking,247,0
bread,585,292
canned,417,0
dairy-eggs,1187,924
fish,355,0
flour,95,0
fresh-meat,306,0
frozen,603,501
grains,200,0
international,184,0


### Found a bug with Rimi discounts - investigate

In [10]:
from scrapers.common import get_html
from bs4 import BeautifulSoup
import json

code = "SH-13"

url = (
    "https://www.rimi.lt/e-parduotuve/lt/"
    "produktai/saldytas-maistas/c/SH-13"
)

params = {
    "currentPage": 1,
    "pageSize": 40,
    "query": f":relevance:allCategories:{code}:assortmentStatus:inAssortment"
}

html = get_html(url, params=params)

soup = BeautifulSoup(html, "html.parser")

cards = soup.select("li.product-grid__item")

print("Cards found:", len(cards))

Cards found: 40


In [11]:
for card in cards:
    if "Greitai užšaldyti žirneliai" in card.get_text():
        print(card.prettify())
        break
else:
    print("Product not found on this page")

<li class="product-grid__item">
 <div class="js-product-container card -horizontal-for-mobile" data-gtm-click-name="Greitai užšaldyti žirneliai RIMI, 400 g" data-gtm-eec-product='{"id":"808614","name":"Greitai u\u017e\u0161aldyti \u017eirneliai RIMI, 400 g","category":"SH-13-5-21","brand":null,"price":1.51,"currency":"EUR"}' data-gtms-banner-title="Greitai užšaldyti žirneliai RIMI, 400 g" data-gtms-click-name="Greitai užšaldyti žirneliai RIMI, 400 g" data-gtms-product-id="808614" data-product-code="808614">
  <a aria-label="Greitai užšaldyti žirneliai RIMI, 400 g" class="card__url js-gtm-eec-product-click" href="/e-parduotuve/lt/produktai/saldytas-maistas/saldytos-darzoves-vaisiai-ir-uogos/saldytos-darzoves-ir-darzoviu-misiniai/greitai-uzsaldyti-zirneliai-rimi-400-g/p/808614">
  </a>
  <div class="label-badges">
  </div>
  <div class="card__image-wrapper">
   <div>
    <img alt="Greitai užšaldyti žirneliai RIMI, 400 g" data-src="https://rimibaltic-res.cloudinary.com/image/upload/b_whit

In [12]:
for card in cards:
    if "Greitai užšaldyti žirneliai" in card.get_text():
        print(card.get_text(separator="\n", strip=True))
        break

1
05
€
2,62 €/kg
Greitai užšaldyti žirneliai RIMI, 400 g
1.51 €
                    už vnt.
1
51
€/vnt.
Kaina už vienetą: 3,77 €/kg
3,77
                        €/kg
Įdėti į krepšelį
Decrease Greitai užšaldyti žirneliai RIMI, 400 g
Increase Greitai užšaldyti žirneliai RIMI, 400 g
Pašalinti Greitai užšaldyti žirneliai RIMI, 400 g


In [13]:
for card in cards:
    if "Greitai užšaldyti žirneliai" in card.get_text():

        product = card.select_one("[data-gtm-eec-product]")

        if product:
            data = json.loads(product["data-gtm-eec-product"])
            print(json.dumps(data, indent=2, ensure_ascii=False))

        break

{
  "id": "808614",
  "name": "Greitai užšaldyti žirneliai RIMI, 400 g",
  "category": "SH-13-5-21",
  "brand": null,
  "price": 1.51,
  "currency": "EUR"
}


### Test new extraction logic

In [14]:

for card in cards:
    if "Greitai užšaldyti žirneliai" in card.get_text():

        major = card.select_one(".price-label__price .major")
        cents = card.select_one(".price-label__price .cents")

        if major and cents:
            promo_price = float(
                f"{major.get_text(strip=True)}."
                f"{cents.get_text(strip=True)}"
            )

            print("Promotional price:", promo_price)

        break


Promotional price: 1.05


In [15]:

for card in cards:
    if "Greitai užšaldyti žirneliai" in card.get_text():

        regular_unit = card.select_one(
            ".card__price-per .sr-only"
        )

        promo_unit = card.select_one(
            ".price-label .price-per-unit"
        )

        print(
            "Regular unit price:",
            regular_unit.get_text(strip=True)
            if regular_unit else None
        )

        print(
            "Promotional unit price:",
            promo_unit.get_text(strip=True)
            if promo_unit else None
        )

        break


Regular unit price: Kaina už vienetą: 3,77 €/kg
Promotional unit price: 2,62 €/kg


In [16]:

import pandas as pd

def select_text(card, selector):
    element = card.select_one(selector)
    return element.get_text(" ", strip=True) if element else None


debug_rows = []

for card in cards:

    product = card.select_one("[data-gtm-eec-product]")

    if product is None:
        continue

    data = json.loads(product["data-gtm-eec-product"])

    debug_rows.append({
        "id": data["id"],
        "name": data["name"],
        "gtm_price": data["price"],
        "card_price": select_text(
            card, ".card__price .sr-only"
        ),
        "label_price": select_text(
            card, ".price-label__price"
        ),
        "regular_unit": select_text(
            card, ".card__price-per .sr-only"
        ),
        "promo_unit": select_text(
            card, ".price-label .price-per-unit"
        ),
        "loyalty_card": bool(card.select_one(
            'img[src*="promotion-labels/rimi-card"]'
        )),
        "badges": select_text(
            card, ".label-badges"
        ),
    })

debug_df = pd.DataFrame(debug_rows)

display(debug_df)


,id,name,gtm_price,card_price,label_price,regular_unit,promo_unit,loyalty_card,badges
0,4013261,"Maskarp. sk. ledai su uogų įdaru DADU, 120 ml",1.09,1.09 €\n už vnt.,NaN,"Kaina už vienetą: 15,57 €/kg",NaN,False,
1,4013250,"Klevų sulos skonio ledai DADU, 150 ml",2.19,2.19 €\n už vnt.,NaN,"Kaina už vienetą: 23,05 €/kg",NaN,False,
2,4013262,"Baltojo persiko skonio ledai DADU, 120 ml",1.09,1.09 €\n už vnt.,NaN,"Kaina už vienetą: 15,57 €/kg",NaN,False,
3,808614,"Greitai užšaldyti žirneliai RIMI, 400 g",1.51,1.51 €\n už vnt.,1 05 €,"Kaina už vienetą: 3,77 €/kg","2,62 €/kg",True,
4,808637,"Pjaustytos žal. šparaginės pupelės RIMI, 400g",1.29,1.29 €\n už vnt.,0 90 €,"Kaina už vienetą: 3,22 €/kg","2,25 €/kg",True,
5,811169,"Greitai užšaldytos šilauogės RIMI, 400 g",3.49,3.49 €\n už vnt.,NaN,"Kaina už vienetą: 8,72 €/kg",NaN,False,
6,808638,"Greitai užšaldyti brokoliai RIMI, 400 g",1.89,1.89 €\n už vnt.,1 29 €,"Kaina už vienetą: 4,72 €/kg","3,22 €/kg",True,
7,811170,"Šaldytas trijų daržovių mišinys RIMI, 400 g",1.19,1.19 €\n už vnt.,NaN,"Kaina už vienetą: 2,97 €/kg",NaN,False,
8,811670,"Šaldytos braškės RIMI, 400 g",1.89,1.89 €\n už vnt.,NaN,"Kaina už vienetą: 4,72 €/kg",NaN,False,
9,811173,"Šald. morkų kub. ir žirn. mišinys RIMI, 400 g",1.09,1.09 €\n už vnt.,NaN,"Kaina už vienetą: 2,72 €/kg",NaN,False,


In [17]:

display(
    debug_df[
        debug_df["regular_unit"].isna()
    ]
)


,id,name,gtm_price,card_price,label_price,regular_unit,promo_unit,loyalty_card,badges


In [18]:
params["currentPage"] = 2

html = get_html(url, params=params)

soup = BeautifulSoup(html, "html.parser")

cards = soup.select("li.product-grid__item")

print("Cards found:", len(cards))

Cards found: 40


In [19]:

import pandas as pd

def select_text(card, selector):
    element = card.select_one(selector)
    return element.get_text(" ", strip=True) if element else None


debug_rows = []

for card in cards:

    product = card.select_one("[data-gtm-eec-product]")

    if product is None:
        continue

    data = json.loads(product["data-gtm-eec-product"])

    debug_rows.append({
        "id": data["id"],
        "name": data["name"],
        "gtm_price": data["price"],
        "card_price": select_text(
            card, ".card__price .sr-only"
        ),
        "label_price": select_text(
            card, ".price-label__price"
        ),
        "regular_unit": select_text(
            card, ".card__price-per .sr-only"
        ),
        "promo_unit": select_text(
            card, ".price-label .price-per-unit"
        ),
        "loyalty_card": bool(card.select_one(
            'img[src*="promotion-labels/rimi-card"]'
        )),
        "badges": select_text(
            card, ".label-badges"
        ),
    })

debug_df = pd.DataFrame(debug_rows)

display(debug_df)


,id,name,gtm_price,card_price,label_price,regular_unit,promo_unit,loyalty_card,badges
0,212298,"Vaniliniai valg. ledai KLASIKA, 1 l / 500 g",3.79,3.79 €\n už vnt.,2 25 €,"Kaina už vienetą: 7,58 €/kg","4,50 €/kg",True,
1,209527,"Vaniliniai valg. ledai BALTIJA, 150 ml / 95 g",2.49,2.49 €\n už vnt.,1 49 €,"Kaina už vienetą: 26,21 €/kg","15,68 €/kg",True,
2,808828,"Vanilės skon. valg. ledai RIMI, 120 ml / 65 g",0.35,0.35 €\n už vnt.,NaN,"Kaina už vienetą: 5,38 €/kg",NaN,False,
3,296505,"Kold. VICI TRADITIONAl, su višt., bar. 400 g",4.39,4.39 €\n už vnt.,NaN,"Kaina už vienetą: 10,97 €/kg",NaN,False,
4,298829,"Koldūnai su jautiena VIČI TRADITIONAL, 400 g",4.39,4.39 €\n už vnt.,NaN,"Kaina už vienetą: 10,97 €/kg",NaN,False,
5,279111,"Van. plombyras su raz. KLASIKA, 120 ml / 70 g",0.89,0.89 €\n už vnt.,NaN,"Kaina už vienetą: 12,71 €/kg",NaN,False,
6,813369,"Šald. menk. filė be od.,be uod. RIMI,MSC,380g",4.99,4.99 €\n už vnt.,2 99 €,"Kaina už vienetą: 13,13 €/kg","7,87 €/kg",True,
7,808706,"Traški 3 sūrių pica su įd. krašt. RIMI, 434 g",4.99,4.99 €\n už vnt.,NaN,"Kaina už vienetą: 11,50 €/kg",NaN,False,
8,223910,"Šaldytų daržovių mišinys HORTEX VIP MIX, 400g",1.79,1.79 €\n už vnt.,NaN,"Kaina už vienetą: 4,47 €/kg",NaN,False,
9,274314,"Valg. ledai ESKIMO KARVUTĖ KLASIKA, 70 ml/60g",1.09,1.09 €\n už vnt.,0 65 €,"Kaina už vienetą: 18,17 €/kg","10,83 €/kg",True,


In [20]:
display(
    debug_df[
        debug_df["label_price"].notna()
        & ~debug_df["loyalty_card"]
    ]
)

,id,name,gtm_price,card_price,label_price,regular_unit,promo_unit,loyalty_card,badges


In [21]:
display(
    debug_df[
        debug_df["regular_unit"].isna()
    ]
)

,id,name,gtm_price,card_price,label_price,regular_unit,promo_unit,loyalty_card,badges


In [22]:
for card in cards:

    regular_unit = card.select_one(
        ".card__price-per .sr-only"
    )

    if regular_unit is None:
        print(card.prettify())
        break
else:
    print("All cards contain regular unit prices")

All cards contain regular unit prices


### Check if rimi was updated correctly

In [23]:
import importlib

from scrapers import rimi

importlib.reload(rimi)

<module 'scrapers.rimi' from '/Users/rokasslivinskas/Projects/end-to-end-AI/eat-easy/notebooks/../scrapers/rimi.py'>

In [24]:
import pandas as pd

test_rows = rimi.scrape(
    "SH-13",
    "saldytas-maistas",
    max_pages=2
)

test_df = pd.DataFrame(test_rows)

print("Products:", len(test_df))
print("Columns:", len(test_df.columns))

rimi page 1: 40 cards, 40 new
rimi page 2: 40 cards, 40 new
Products: 80
Columns: 15


In [25]:
display(
    test_df[test_df["id"] == "808614"][
        [
            "name",
            "price",
            "regular_price",
            "unit_price",
            "regular_unit_price",
            "unit",
            "on_promo",
            "loyalty_card_required",
            "category"
        ]
    ]
)

,name,price,regular_price,unit_price,regular_unit_price,unit,on_promo,loyalty_card_required,category
3,"Greitai užšaldyti žirneliai RIMI, 400 g",1.05,1.51,2.62,3.77,kg,True,True,saldytas-maistas/saldytos-darzoves-vaisiai-ir-...


In [26]:
print("Total products:", len(test_df))

print("Promotions:",
      test_df["on_promo"].sum())

print("Missing unit prices:",
      test_df["unit_price"].isna().sum())

print("Missing regular unit prices:",
      test_df["regular_unit_price"].isna().sum())

print("Inconsistent promotions:",
      (
          test_df["on_promo"]
          & (test_df["price"] >= test_df["regular_price"])
      ).sum())

print("Loyalty card without detected discount:",
      (
          test_df["loyalty_card_required"]
          & ~test_df["on_promo"]
      ).sum())

Total products: 80
Promotions: 21
Missing unit prices: 0
Missing regular unit prices: 0
Inconsistent promotions: 0
Loyalty card without detected discount: 1


In [27]:
display(
    test_df[test_df["on_promo"]][
        [
            "name",
            "price",
            "regular_price",
            "unit_price",
            "regular_unit_price",
            "unit",
            "loyalty_card_required"
        ]
    ]
)

,name,price,regular_price,unit_price,regular_unit_price,unit,loyalty_card_required
3,"Greitai užšaldyti žirneliai RIMI, 400 g",1.05,1.51,2.62,3.77,kg,True
4,"Pjaustytos žal. šparaginės pupelės RIMI, 400g",0.90,1.29,2.25,3.22,kg,True
6,"Greitai užšaldyti brokoliai RIMI, 400 g",1.29,1.89,3.22,4.72,kg,True
10,"Greitai užšald. žiediniai kopūstai RIMI, 400g",0.97,1.39,2.42,3.47,kg,True
16,"Šald. krev. be kiautų RIMI, ASC, 31/40, 400 g",5.39,8.99,13.47,22.47,kg,True
17,"Šald. krevetės NOWACO EXLUSIVE, ASC, 300 g",5.99,9.99,19.97,33.30,kg,True
19,"Šald. blyšk. krev. be k. RIMI, ASC, 200 g",2.63,4.39,13.15,21.95,kg,True
21,"Sojų pupelės RIMI PLANET EDAMAME, 450 g",2.79,4.19,6.20,9.31,kg,True
28,"Šald. matjes silkių filė be odos NOWACO, 350g",4.61,7.69,13.17,21.97,kg,True
30,"Šaldytos avietės RIMI, 400 g",3.59,5.19,8.97,12.97,kg,True


In [28]:
display(
    test_df[
        test_df["loyalty_card_required"]
        & ~test_df["on_promo"]
    ].T
)

,51
store,rimi
id,803757
name,"Sviest. mielinė tešla SELECTION BY RIMI, 400g"
brand,RIMI
price,5.29
regular_price,3.59
loyalty_card_required,True
on_promo,False
unit_price,8.97
regular_unit_price,8.97
